<a href="https://colab.research.google.com/github/sohansaw1210-byte/chief-of-staff-agent/blob/main/Homingo_Hackathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
inbox = [
    {"id": 1, "from": "MSEDCL", "subject": "Electricity bill due",
     "body": "Your bill of Rs 1,840 is due on 28 Nov 2026. Pay online to avoid a late fee."},
    {"id": 2, "from": "Dr. Mehta's Clinic", "subject": "Appointment reminder",
     "body": "Please confirm your check-up. Available slots: Tue 24 Nov 5pm, Wed 25 Nov 11am."},
    {"id": 3, "from": "ShopNow", "subject": "Refund update",
     "body": "Your refund of Rs 2,499 for order #8841 is still pending. Please contact support."},
]

for mail in inbox:
    print(mail["id"], "|", mail["from"], "|", mail["subject"])

1 | MSEDCL | Electricity bill due
2 | Dr. Mehta's Clinic | Appointment reminder
3 | ShopNow | Refund update


In [4]:
!pip install -q google-genai

import json
import time
from google import genai
from google.genai import types
from google.colab import userdata
from google.genai.errors import ServerError

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
MODEL = "gemini-3.8-flash"

PROMPT = """You are a careful personal chief of staff. Read the email and decide what needs doing.
Reply ONLY with JSON in this format:
{"action": "pay_bill" or "book_appointment" or "chase_refund" or "none",
 "summary": "one short sentence",
 "deadline": "date if any, else null",
 "draft": "a short message or reminder you would send, else null"}
Never claim an action is done. You only propose."""

def analyze(mail, retries=3, delay=2):
    text = f"From: {mail['from']}\nSubject: {mail['subject']}\n\n{mail['body']}"
    for attempt in range(retries):
        try:
            resp = client.models.generate_content(
                model=MODEL,
                contents=PROMPT + "\n\nEMAIL:\n" + text,
                config=types.GenerateContentConfig(response_mime_type="application/json"),
            )
            return json.loads(resp.text)
        except ServerError as e:
            if e.code == 503 and attempt < retries - 1:
                print(f"[Warning] 503 Service Unavailable. Retrying in {delay}s...")
                time.sleep(delay)
                delay *= 2
            else:
                raise e

for mail in inbox:
    try:
        result = analyze(mail)
        print(f"--- Email {mail['id']}: {mail['subject']}")
        print(json.dumps(result, indent=2))
    except Exception as e:
        print(f"--- Email {mail['id']}: Failed to analyze due to error: {e}")

[Warning] 503 Service Unavailable. Retrying in 2s...
[Warning] 503 Service Unavailable. Retrying in 4s...
--- Email 1: Electricity bill due
{
  "action": "pay_bill",
  "summary": "Pay the MSEDCL electricity bill of Rs 1,840 due on 28 Nov 2026.",
  "deadline": "28 Nov 2026",
  "draft": "Reminder: Pay the MSEDCL electricity bill of Rs 1,840 before 28 Nov 2026 to avoid late fees."
}
--- Email 2: Appointment reminder
{
  "action": "book_appointment",
  "summary": "Dr. Mehta's clinic requires confirmation for an upcoming check-up appointment.",
  "deadline": null,
  "draft": "Hello, I would like to confirm the check-up appointment for Tuesday, November 24 at 5:00 PM. Thank you."
}
--- Email 3: Refund update
{
  "action": "chase_refund",
  "summary": "Follow up with ShopNow support regarding a pending refund of Rs 2,499 for order #8841.",
  "deadline": null,
  "draft": "Hello ShopNow Support, I received an update that my refund of Rs 2,499 for order #8841 is pending. Could you please check t

In [5]:
log = []  # a record of everything the agent did

def approve_and_run(mail, result):
    if result["action"] == "none":
        print(f"Email {mail['id']}: nothing to do.\n")
        return

    print("=" * 50)
    print(f"Email {mail['id']} | {mail['from']} | {mail['subject']}")
    print(f"What I found : {result['summary']}")
    print(f"Deadline     : {result['deadline']}")
    print(f"Proposed     : {result['action']}")
    print(f"Draft        : {result['draft']}")

    answer = input("Approve this? (yes / no): ").strip().lower()

    if answer == "yes":
        # SIMULATED for now. Nothing is really sent or paid.
        print(f"✅ Done (simulated): {result['action']}\n")
        log.append({"email": mail["id"], "action": result["action"], "status": "approved"})
    else:
        print("⏭️ Skipped. Nothing was done.\n")
        log.append({"email": mail["id"], "action": result["action"], "status": "rejected"})

# Analyze once, then ask for approval on each
results = [(mail, analyze(mail)) for mail in inbox]

for mail, result in results:
    approve_and_run(mail, result)

print("ACTIVITY LOG")
for entry in log:
    print(entry)

[Warning] 503 Service Unavailable. Retrying in 2s...
[Warning] 503 Service Unavailable. Retrying in 2s...
Email 1 | MSEDCL | Electricity bill due
What I found : Pay the MSEDCL electricity bill of Rs 1,840.
Deadline     : 28 Nov 2026
Proposed     : pay_bill
Draft        : Reminder: The MSEDCL electricity bill of Rs 1,840 is due on 28 Nov 2026. Please pay online to avoid late fees.
Approve this? (yes / no): yes
✅ Done (simulated): pay_bill

Email 2 | Dr. Mehta's Clinic | Appointment reminder
What I found : Confirm a check-up appointment slot with Dr. Mehta's Clinic.
Deadline     : None
Proposed     : book_appointment
Draft        : Hello, I would like to confirm the appointment for Tuesday, 24 Nov at 5pm. Thank you.
Approve this? (yes / no): no
⏭️ Skipped. Nothing was done.

Email 3 | ShopNow | Refund update
What I found : ShopNow requested contacting support regarding a pending refund of Rs 2,499 for order #8841.
Deadline     : None
Proposed     : chase_refund
Draft        : Hi Support,